# Face Recognition Demo 

This notebook compares a target face against a small set of other faces from the LFW dataset using DeepFace embeddings.

In [ ]:
!pip install deepface scikit-learn matplotlib opencv-python-headless --quiet

In [ ]:
from deepface import DeepFace
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people


In [ ]:
# Load LFW data
min_faces_per_person = 40
lfw_people = fetch_lfw_people(min_faces_per_person=min_faces_per_person, resize=0.4)

print("Number of images in the dataset:", lfw_people.data.shape[0])
print("Number of people in the dataset:", len(np.unique(lfw_people.target)))
print("Number of features in each image:", lfw_people.data.shape[1])


In [ ]:
def ensure_rgb(image):
    """DeepFace expects RGB images with three channels; LFW images are often grayscale."""
    if len(image.shape) == 2:
        image = np.repeat(image[:, :, None], 3, axis=2)
    return image

def resize(image, window_height=500):
    aspect_ratio = float(image.shape[1]) / float(image.shape[0])
    window_width = window_height / aspect_ratio
    image = cv2.resize(image, (int(window_height), int(window_width)))
    return image


In [ ]:
target_name = "Arnold Schwarzenegger"
target_index = np.where(lfw_people.target_names == target_name)[0][0]
print(f"Index of {target_name} in target_names: {target_index}")

target_indices = np.where(lfw_people.target == target_index)[0]
print(f"Indices of all images of {target_name}: {target_indices}")

first_target_face = ensure_rgb(lfw_people.images[target_indices[0]])

num_target_images = 5
target_faces = np.stack([ensure_rgb(face) for face in lfw_people.images[target_indices[:num_target_images]]])

other_indices = np.where(lfw_people.target != target_index)[0]
other_faces = np.stack([ensure_rgb(face) for face in lfw_people.images[other_indices[:num_target_images]]])

joined_data = [(face, target_name) for face in target_faces]
other_labels = [lfw_people.target_names[lfw_people.target[idx]] for idx in other_indices[:num_target_images]]
joined_data += list(zip(other_faces, other_labels))

np.random.shuffle(joined_data)
joined_faces = np.stack([item[0] for item in joined_data])
joined_labels = [item[1] for item in joined_data]


### Determine matches with Cosine Similarity

The cosine similarity is a measure of the angle between the two vectors. A value of 1.0 means the vectors are identical, while 0.0 means they are orthogonal. To compute the cosine similarity, divide the dot product of the two vectors by the product of their magnitudes.

* The dot product of the two vectors means multiplying each corresponding element of the two vectors and summing the results.
* The magnitude of a vector is the square root of the sum of the squares of its elements.

The sum of the squares of the elements of a vector is also known as the L2 norm or Euclidean norm. More on norms here: https://www.geeksforgeeks.org/maths/vector-norms/


In [ ]:
model_list = ["VGG-Face", "Facenet", "OpenFace", "DeepFace", "Dlib", "ArcFace"]
model_index = 0

reference_face_encoding = DeepFace.represent(
    first_target_face,
    model_name=model_list[model_index],
    enforce_detection=False,
)[0]["embedding"]

joined_faces_encodings = [
    DeepFace.represent(face, model_name=model_list[model_index], enforce_detection=False)[0]["embedding"]
    for face in joined_faces
]

results = []
for encoding in joined_faces_encodings:
    # Cosine similarity measures how aligned two face embeddings are.
    # A value near 1.0 means the faces are very similar, while values closer to 0
    # indicate they are less similar to the reference face.
    similarity = np.dot(reference_face_encoding, encoding) / (
        np.linalg.norm(reference_face_encoding) * np.linalg.norm(encoding)
    )
    results.append(similarity)

threshold = 0.5
print("Similarity scores:", results)


In [ ]:
num_images = len(joined_faces)
plt.figure(figsize=(15, 5))
for i in range(num_images):
    ax = plt.subplot(2, num_images // 2, i + 1)
    ax.imshow(joined_faces[i], cmap='gray')
    ax.set_title(f"Similarity: {results[i]:.2f}\nMatch: {'Yes' if results[i] > threshold else 'No'}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel(joined_labels[i], labelpad=10)

plt.tight_layout()
plt.show()
